Load the results dataset to run evaluation on it.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('DUSH_HFT')
# WANDB_TOKEN = userdata.get('DUSH_WANDB')

In [ ]:
# Login into Hugging Face Hub
from huggingface_hub import login
login(token=HF_TOKEN)

# import wandb
# wandb.login(key=WANDB_TOKEN)

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "dinushiTJ/nz_hansard_inference_results",
    split="test"
)


First we need to make sure there were no errored responses given by the LLMs during inference. If a model failed to provide a valid JSON output, that must be kept out from being evaluated.

In [ ]:
import json

def count_json_ok_false(dataset, column_name):
    """
    Counts the number of entries in a dataset where the 'json_ok' field
    within a specified JSON column is False.

    Args:
        dataset (datasets.Dataset): The Hugging Face dataset.
        column_name (str): The name of the column containing JSON strings.

    Returns:
        int: The number of entries where 'json_ok' is False.
    """
    false_count = 0
    for item in dataset:
        try:
            raw_output = item[column_name]
            parsed_json = json.loads(raw_output)
            if 'json_ok' in parsed_json and not parsed_json['json_ok']:
                false_count += 1
        except json.JSONDecodeError:
            print(f"Warning: Could not decode JSON from item in column '{column_name}': {raw_output[:100]}...")
            false_count += 1 # Consider malformed JSON as not 'ok'
        except KeyError:
            print(f"Warning: 'json_ok' key not found in item in column '{column_name}': {parsed_json}")
            false_count += 1 # Consider missing 'json_ok' as not 'ok'
    return false_count


In [ ]:
false_count_base_response = count_json_ok_false(ds, "BaseResponse")
print(f"Number of 'json_ok: false' in 'BaseResponse' column: {false_count_base_response}")

false_count_base4b_response = count_json_ok_false(ds, "Base4BResponse")
print(f"Number of 'json_ok: false' in 'Base4BResponse' column: {false_count_base4b_response}")

false_count_ft_response = count_json_ok_false(ds, "FTResponse")
print(f"Number of 'json_ok: false' in 'FTResponse' column: {false_count_ft_response}")

false_count_ft4b_response = count_json_ok_false(ds, "FT4BResponse")
print(f"Number of 'json_ok: false' in 'FT4BResponse' column: {false_count_ft4b_response}")

Number of 'json_ok: false' in 'BaseResponse' column: 0
Number of 'json_ok: false' in 'Base4BResponse' column: 0
Number of 'json_ok: false' in 'FTResponse' column: 0
Number of 'json_ok: false' in 'FT4BResponse' column: 0


In [ ]:
import json

def extract_label(example, column_name):
    """
    Extracts the 'label' from the JSON content of a specified column in a dataset example.

    Args:
        example (dict): A single dictionary representing a row from the Hugging Face dataset.
        column_name (str): The name of the column containing JSON strings.

    Returns:
        str: The extracted label as a string, or 'invalid_label' if an error occurs or label is not found.
    """
    try:
        raw_output = example[column_name]
        parsed_json = json.loads(raw_output)
        label = parsed_json.get('output', {}).get('label') # Access 'output' then 'label'
        if label is None:
            # If 'label' is not directly under 'output', try direct access (for older formats or variations)
            label = parsed_json.get('label')
        return str(label) if label is not None else 'invalid_label'
    except (json.JSONDecodeError, KeyError) as e:
        return 'invalid_label'
    except Exception as e:
        # unknown error
        return 'invalid_label'

In [ ]:
# Example usage:
# Let's get the first example from your dataset
first_example = ds[0]

# Extract label from 'BaseResponse' column
label_from_base = extract_label(first_example, "BaseResponse")
print(f"Label from 'BaseResponse' for first example: {label_from_base}")

# Extract label from 'FTResponse' column
label_from_ft = extract_label(first_example, "FTResponse")
print(f"Label from 'FTResponse' for first example: {label_from_ft}")

# Let's try with a column that might not have a label or valid JSON (for demonstration)
# Assuming 'HpsProceedingContent' doesn't contain a JSON label
label_from_content = extract_label(first_example, "HpsProceedingContent")
print(f"Label from 'HpsProceedingContent' (expected None): {label_from_content}")


Label from 'BaseResponse' for first example: maori_origin | non_maori_origin
Label from 'FTResponse' for first example: non_maori_origin
Label from 'HpsProceedingContent' (expected None): invalid_label


Eval function

In [ ]:
import numpy as np

def get_active_labels(y_true, y_pred, label_map):
    active_ids = sorted(set(y_true) | set(y_pred))
    active_names = [
        name for name, idx in label_map.items()
        if idx in active_ids
    ]
    return active_ids, active_names

In [ ]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

def evaluate(column_name: str, model_type: str):
    # Task labels only
    LABEL_MAP = {
        "maori_origin": 0,
        "non_maori_origin": 1,
    }
    INVALID_ID = -1  # sentinel, NOT a class

    y_true_all = []
    y_pred_all = []

    # ---------- collect predictions ----------
    if model_type == "embedding":
        for yt, yp in zip(ds["classification_label"], ds[column_name]):
            y_true_all.append(LABEL_MAP[yt])
            y_pred_all.append(LABEL_MAP.get(yp, INVALID_ID))

    elif model_type == "llm":
        for item in ds:
            true_label = LABEL_MAP[item["classification_label"]]
            pred_label_str = extract_label(item, column_name)
            pred_label = LABEL_MAP.get(pred_label_str, INVALID_ID)

            y_true_all.append(true_label)
            y_pred_all.append(pred_label)

    else:
        raise ValueError("model_type must be 'embedding' or 'llm'")

    # ---------- validity analysis ----------
    total = len(y_pred_all)
    valid_mask = [p != INVALID_ID for p in y_pred_all]

    valid_count = sum(valid_mask)
    invalid_count = total - valid_count
    valid_rate = valid_count / total

    # ---------- filter for task metrics ----------
    y_true = [yt for yt, m in zip(y_true_all, valid_mask) if m]
    y_pred = [yp for yp, m in zip(y_pred_all, valid_mask) if m]

    print(f"\nValid prediction rate: {valid_rate:.3f} ({valid_count}/{total})")

    # ---------- total failure case ----------
    if valid_count == 0:
        print("\n⚠️ No valid task predictions produced by the model.")
        return {
            "valid_rate": valid_rate,
            "accuracy": 0.0,
            "f1_macro": 0.0,
            "f1_weighted": 0.0,
            "note": "Model failed to produce any valid labels",
        }

    # ---------- task metrics ----------
    labels = list(LABEL_MAP.values())
    target_names = list(LABEL_MAP.keys())

    metrics = {
        "valid_rate": valid_rate,
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(
            y_true, y_pred,
            labels=labels,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            y_true, y_pred,
            labels=labels,
            average="weighted",
            zero_division=0,
        ),
    }

    print("\nConfusion Matrix (task labels only)")
    print(confusion_matrix(y_true, y_pred, labels=labels))

    print("\nClassification Report (task labels only)")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=labels,
            target_names=target_names,
            digits=4,
            zero_division=0,
        )
    )

    return metrics


In [ ]:
evaluate(column_name="BaseEmbGemZeroLabel", model_type="embedding")


Valid prediction rate: 1.000 (213/213)

Confusion Matrix (task labels only)
[[ 21  30]
 [  9 153]]

Classification Report (task labels only)
                  precision    recall  f1-score   support

    maori_origin     0.7000    0.4118    0.5185        51
non_maori_origin     0.8361    0.9444    0.8870       162

        accuracy                         0.8169       213
       macro avg     0.7680    0.6781    0.7027       213
    weighted avg     0.8035    0.8169    0.7987       213



{'valid_rate': 1.0,
 'accuracy': 0.8169014084507042,
 'f1_macro': 0.7027375201288244,
 'f1_weighted': 0.7987389716722232}

In [ ]:
evaluate(column_name="FTEmbGemZeroLabel", model_type="embedding")


Valid prediction rate: 1.000 (213/213)

Confusion Matrix (task labels only)
[[ 37  14]
 [  5 157]]

Classification Report (task labels only)
                  precision    recall  f1-score   support

    maori_origin     0.8810    0.7255    0.7957        51
non_maori_origin     0.9181    0.9691    0.9429       162

        accuracy                         0.9108       213
       macro avg     0.8995    0.8473    0.8693       213
    weighted avg     0.9092    0.9108    0.9077       213



{'valid_rate': 1.0,
 'accuracy': 0.9107981220657277,
 'f1_macro': 0.8693209338370629,
 'f1_weighted': 0.9076873329485778}

In [ ]:
evaluate(column_name="Base4BResponse", model_type="llm")


Valid prediction rate: 1.000 (213/213)

Confusion Matrix (task labels only)
[[ 34  17]
 [ 28 134]]

Classification Report (task labels only)
                  precision    recall  f1-score   support

    maori_origin     0.5484    0.6667    0.6018        51
non_maori_origin     0.8874    0.8272    0.8562       162

        accuracy                         0.7887       213
       macro avg     0.7179    0.7469    0.7290       213
    weighted avg     0.8062    0.7887    0.7953       213



{'valid_rate': 1.0,
 'accuracy': 0.7887323943661971,
 'f1_macro': 0.7289999717266533,
 'f1_weighted': 0.7953029608565471}

In [ ]:
evaluate(column_name="FT4BResponse", model_type="llm")


Valid prediction rate: 0.991 (211/213)

Confusion Matrix (task labels only)
[[ 43   7]
 [ 13 148]]

Classification Report (task labels only)
                  precision    recall  f1-score   support

    maori_origin     0.7679    0.8600    0.8113        50
non_maori_origin     0.9548    0.9193    0.9367       161

        accuracy                         0.9052       211
       macro avg     0.8613    0.8896    0.8740       211
    weighted avg     0.9105    0.9052    0.9070       211



{'valid_rate': 0.9906103286384976,
 'accuracy': 0.9052132701421801,
 'f1_macro': 0.8740148077382375,
 'f1_weighted': 0.9069960394224054}

In [ ]:
evaluate(column_name="BaseResponse", model_type="llm")


Valid prediction rate: 0.000 (0/213)

⚠️ No valid task predictions produced by the model.


{'valid_rate': 0.0,
 'accuracy': 0.0,
 'f1_macro': 0.0,
 'f1_weighted': 0.0,
 'note': 'Model failed to produce any valid labels'}

In [ ]:
evaluate(column_name="FTResponse", model_type="llm")


Valid prediction rate: 1.000 (213/213)

Confusion Matrix (task labels only)
[[ 11  40]
 [ 32 130]]

Classification Report (task labels only)
                  precision    recall  f1-score   support

    maori_origin     0.2558    0.2157    0.2340        51
non_maori_origin     0.7647    0.8025    0.7831       162

        accuracy                         0.6620       213
       macro avg     0.5103    0.5091    0.5086       213
    weighted avg     0.6429    0.6620    0.6517       213



{'valid_rate': 1.0,
 'accuracy': 0.6619718309859155,
 'f1_macro': 0.5085875416559856,
 'f1_weighted': 0.6516602821233991}